# 07 — Results and recommendations

One table across every method, then what is and is not recommended. Labels used in every notebook: **MEASURED** (a real provider or real run produced the number), **ESTIMATED** (derived here from measured data by a stated formula), **SIMULATED** (mock runner, no real model), **NOT_RUN**, **UNKNOWN** (the evidence does not say). Implementation and wiring are reported separately from evaluation: code that exists is not a measured capability.

In [1]:
import json
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 11  # the pre-registered bootstrap seed; reused for every derived computation here
random.seed(SEED)
np.random.seed(SEED)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "EVIDENCE_MAP.md").exists())  # research/council-ablation
EV = ROOT / "evidence"
JEV = ROOT.parent / "jev" / "evidence"  # the JEV study's canonical copy in this repository
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)


def load(rel):
    return json.loads((EV / rel).read_text())


def jsonl(rel):
    return [json.loads(x) for x in (EV / rel).read_text().splitlines() if x.strip()]


def final_calls(rel):
    """The harness cache is append-only: a re-asked call (Amendments 1 and 2) appends a new row and the
    last row per (member, case) is the one analysed. Earlier rows stay in the file as recorded calls."""
    return pd.DataFrame(jsonl(rel)).drop_duplicates(["member", "case_id"], keep="last").reset_index(drop=True)


R = load("council/results.json")
REG = {r["id"]: r for r in load("reasoning_methods/reasoning_methods_registry.json")}
print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__, "| seed", SEED)
print("evidence files:", sorted(str(p.relative_to(ROOT)) for p in EV.rglob("*") if p.is_file()))


python 3.12.12 | pandas 3.1.0 | numpy 2.5.4 | seed 11
evidence files: ['evidence/SOURCE_MANIFEST.json', 'evidence/council/calibration_by_member.csv', 'evidence/council/cases.csv', 'evidence/council/cases.json', 'evidence/council/conditions.csv', 'evidence/council/cost_latency_by_condition.csv', 'evidence/council/judge_calls.csv', 'evidence/council/judge_calls.jsonl', 'evidence/council/member_calls.csv', 'evidence/council/member_calls.jsonl', 'evidence/council/member_calls_budget300_archive.csv', 'evidence/council/member_calls_budget300_archive.jsonl', 'evidence/council/results.json', 'evidence/decision_methods/ablation-2026-10-20.json', 'evidence/notebook_runs.json', 'evidence/publication_scan.json', 'evidence/reasoning_methods/method_status_summary.csv', 'evidence/reasoning_methods/reasoning_methods_registry.json', 'evidence/reasoning_methods/reasoning_strategies_mock_benchmark.md', 'evidence/reasoning_methods/research_plans.json', 'evidence/replay_check.json']


In [2]:
#: Evaluation state per method, with the evidence that sets it. Explicit, not inferred from text.
EVALUATION = {
    "deterministic-baseline": "MEASURED",      # reference arm of the decision-methods ablation
    "single-strong-model": "MEASURED",         # condition A of the council benchmark
    "council": "MEASURED",                     # conditions B1-D, F (harness, not the product endpoint)
    "proposer-judge": "MEASURED",              # as condition E (judge over C) in the harness only
    "critic-veto": "NOT_RUN",
    "critic-selective-council": "NOT_RUN",
    "cot": "SIMULATED (mock runner)",
    "tot": "SIMULATED (mock runner)",
    "got": "SIMULATED (mock runner)",
    "self-consistency": "NOT_RUN",
    "react": "NOT_RUN",
    "reflexion": "NOT_RUN",
    "pot-evidence": "NOT_RUN",
    "mcts": "NOT_RUN",
    "jev": "MEASURED",                         # real provider calls on decision-corpus-v1
    "agentic-llm-decision": "MEASURED",        # decision-methods ablation, small local model
    "hermes-agent-patterns": "NOT_RUN",
}


def status_row(method_id):
    """Implementation / wiring / evaluation labels, derived from the registry snapshot only."""
    r = REG[method_id]
    implemented = "NOT_IMPLEMENTED" if r["wiring"] == "NOT_PRESENT" else "IMPLEMENTED"
    wired = "WIRED" if r["wiring"] in {"WIRED", "DECISION_AFFECTING", "OBSERVED", "EVALUATED"} else "NOT_WIRED"
    evaluation = EVALUATION[method_id]
    return {"method": r["name"], "implemented": implemented, "wired": wired, "authority": r["authority"],
            "evaluation": evaluation, "production": r["production"], "recommendation": r["recommendation"],
            "result": r["result"], "limitations": r["limitations"]}


In [3]:
ids = ["deterministic-baseline", "single-strong-model", "council", "proposer-judge", "critic-veto", "cot", "tot", "got",
       "self-consistency", "react", "reflexion", "pot-evidence", "mcts", "jev", "agentic-llm-decision", "hermes-agent-patterns"]
summary = pd.DataFrame([status_row(i) for i in ids]).set_index("method")
summary.to_csv(EV / "reasoning_methods" / "method_status_summary.csv")
summary[["implemented", "wired", "authority", "evaluation", "production", "recommendation"]]

,implemented,wired,authority,evaluation,production,recommendation
method,,,,,,
Deterministic Athena baseline,IMPLEMENTED,WIRED,DECIDES,MEASURED,IN_PRODUCT,ADOPT
Single strong reasoning model (reasoning-depth router),IMPLEMENTED,WIRED,ADVISORY,MEASURED,IN_PRODUCT,ADOPT
Council (multi-persona / multi-model vote),IMPLEMENTED,WIRED,ADVISORY,MEASURED,IN_PRODUCT,SELECTIVE
Proposer / Judge (training champion re-evaluation),IMPLEMENTED,WIRED,DECIDES,MEASURED,IN_PRODUCT_OFF_BY_DEFAULT,SELECTIVE
Semantic critic with protective-minority veto,NOT_IMPLEMENTED,NOT_WIRED,NONE,NOT_RUN,NOT_IN_PRODUCT,RESEARCH
Chain-of-Thought (Chain Aggregation),IMPLEMENTED,WIRED,CONFIDENCE_ONLY,SIMULATED (mock runner),API_ONLY,RESEARCH
Tree-of-Thoughts (Tree Aggregation),IMPLEMENTED,WIRED,CONFIDENCE_ONLY,SIMULATED (mock runner),API_ONLY,RESEARCH
Graph-of-Thoughts (Graph Consensus) + preprocessing GoT trace,IMPLEMENTED,WIRED,CONFIDENCE_ONLY,SIMULATED (mock runner),API_ONLY,RESEARCH
Self-Consistency,NOT_IMPLEMENTED,NOT_WIRED,NONE,NOT_RUN,NOT_IN_PRODUCT,RESEARCH


In [4]:
pd.Series(R["decisions"], name="council decision (pre-registered rule)").to_frame()

,council decision (pre-registered rule)
B1_same_provider_roles,REJECTED
B2_same_provider_sampling,REJECTED
C_multi_provider,REJECTED
D_characteristic_diverse,REJECTED
E_judge,REJECTED
F_adaptive_routing,REJECTED


## Recommendations

1. **Default path:** a single strong model plus deterministic checks and the evidence validator. Measured: no council beat it on this set.
2. **Council:** not a default. `REJECTED_ON_THIS_BENCHMARK`; the next study must first show solo headroom (≥ 15 % error on calibration) or stop as `NOT_RUN — CEILING`.
3. **Critic veto:** the most promising hypothesis (the critic was right and outvoted); it needs its own pre-registration before any claim.
4. **CoT / ToT / GoT:** confidence and tie-break behaviour only; no real-provider measurement. No capability claim.
5. **ReAct, MCTS, self-consistency:** not implemented. **Reflexion, PoT:** wired, not evaluated.
6. **Jev:** measured on one decision family; gain with semantics not established (intervals overlap); cost unknown.

Every untested method is recorded as `RESEARCH_PLANNED` with a hypothesis, baseline, metric, budget, stopping rule and production gate (`METHODOLOGY.md`). None is presented as a product capability.

## Limitations

Ceiling effect (70/71); two provider families, the second much weaker; harness calls models directly (not the product path, no validator, no tenant context); one provider-enforced temperature; the case author is also the harness author. See `LIMITATIONS.md`.